# ARS Session 3 — Adaptive Steering Router

This experiment compares three isolated arms on the same deterministic GSM8K subset: untouched Phi-2, the fixed V2 checkpoint, and a new question-conditioned router. It does not modify the Session 2 notebook or checkpoint. Phi-2 remains frozen.

Attach the verified V2 checkpoint and its p99 calibration when available. Existing rollout shards and a compatible steering bank are reused automatically. If rollout shards are absent, the notebook runs Gate 0 and collects fresh shards. If V2 artifacts are absent, it can calibrate and train a replacement V2 arm from those rollouts. Phase failures are recorded so the notebook reaches its final status cell.


## 1. Execution configuration

Edit the artifact paths to match your Kaggle input dataset.


In [ ]:
# ==========================================
# EXECUTION & SCALE CONFIGURATION
# ==========================================
import os
import torch
from pathlib import Path
from datetime import datetime

SMOKE_TEST = False
AUTO_COLLECT_IF_MISSING = True
AUTO_BOOTSTRAP_V2_IF_MISSING = True
ATTENTION_IMPLEMENTATION = "sdpa"
ARTIFACT_INPUT_DIR = Path("/kaggle/input/ars-session2-artifacts")
V2_CHECKPOINT = ARTIFACT_INPUT_DIR / "steernet_v2.pt"
V2_NORM_CALIBRATION = ARTIFACT_INPUT_DIR / "norms_v2.json"
ROLLOUT_DIR = Path("/kaggle/input/ars-session1-rollouts")
V2_ROUTE = [12, 15, 18, 21]
CANDIDATE_LAYERS = [8, 11, 12, 13, 15, 17, 18, 19, 20, 21, 22, 23, 24, 26]
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = Path("/kaggle/working/ars_session3") / RUN_ID
BOOTSTRAP_V2_CHECKPOINT = OUTPUT_DIR / "session2_steernet_v2.pt"
BOOTSTRAP_V2_NORM_CALIBRATION = OUTPUT_DIR / "session2_norms_v2.json"
BANK_CHECKPOINT_OVERRIDE = None  # Set to a Kaggle input path to select a specific bank checkpoint.
BANK_ROOTS = [Path("/kaggle/working/ars_session3"), Path("/kaggle/input")]
def is_compatible_bank(path):
    try:
        state = torch.load(path, map_location="cpu", weights_only=True)
        return (state.get("architecture") == "steernet_v2" and
                state.get("candidate_layers") == CANDIDATE_LAYERS)
    except (OSError, RuntimeError, ValueError, TypeError, AttributeError):
        return False
bank_candidates = []
for bank_root in BANK_ROOTS:
    if bank_root.exists():
        bank_candidates.extend(bank_root.glob("**/session3_steering_bank.pt"))
bank_candidates = [path for path in bank_candidates if is_compatible_bank(path)]
if BANK_CHECKPOINT_OVERRIDE is not None and is_compatible_bank(Path(BANK_CHECKPOINT_OVERRIDE)):
    BANK_CHECKPOINT = Path(BANK_CHECKPOINT_OVERRIDE)
    BANK_REUSED = True
elif bank_candidates:
    BANK_CHECKPOINT = max(bank_candidates, key=lambda path: path.stat().st_mtime)
    BANK_REUSED = True
else:
    BANK_CHECKPOINT = OUTPUT_DIR / "session3_steering_bank.pt"
    BANK_REUSED = False
ROUTER_CHECKPOINT = OUTPUT_DIR / "session3_router.pt"
EXPANDED_NORM_CALIBRATION = OUTPUT_DIR / "candidate_norms_p99.json"

if SMOKE_TEST:
    CALIB_N, GATE0_QUESTIONS, COLLECTION_QUESTIONS, COLLECTION_K = 24, 10, 40, 2
    V2_BOOTSTRAP_EPOCHS = 1
    BANK_EPOCHS, MAX_RFT_QUESTIONS = 1, 40
    BANK_ROUTE_EXAMPLES_PER_LAYER, BANK_SCREEN_ROWS = 8, 8
    MAX_PAIR_ROUTES, MAX_TRIPLE_ROUTES, MAX_QUAD_ROUTES = 3, 2, 1
    MAX_ROUTER_ROWS, ROUTER_EPOCHS = 12, 1
    ROUTE_COST = 0.01
    BENCH_N_TEST, BENCH_SHOTS = 24, [0]
else:
    CALIB_N, GATE0_QUESTIONS, COLLECTION_QUESTIONS, COLLECTION_K = 128, 60, 2000, 8
    V2_BOOTSTRAP_EPOCHS = 2
    BANK_EPOCHS, MAX_RFT_QUESTIONS = 2, 2000
    BANK_ROUTE_EXAMPLES_PER_LAYER, BANK_SCREEN_ROWS = 256, 24
    MAX_PAIR_ROUTES, MAX_TRIPLE_ROUTES, MAX_QUAD_ROUTES = 12, 8, 6
    MAX_ROUTER_ROWS, ROUTER_EPOCHS = 128, 2
    ROUTE_COST = 0.01
    BENCH_N_TEST, BENCH_SHOTS = 800, [0, 2]

print({"smoke_test": SMOKE_TEST, "fixed_route": V2_ROUTE,
       "candidate_layers": CANDIDATE_LAYERS, "calibration_n": CALIB_N,
       "gate0_n": GATE0_QUESTIONS, "auto_bootstrap_v2": AUTO_BOOTSTRAP_V2_IF_MISSING,
       "bank_epochs": BANK_EPOCHS, "route_examples_per_layer": BANK_ROUTE_EXAMPLES_PER_LAYER,
       "route_caps": [MAX_PAIR_ROUTES, MAX_TRIPLE_ROUTES, MAX_QUAD_ROUTES],
       "router_epochs": ROUTER_EPOCHS,
       "benchmark_n": BENCH_N_TEST, "shots": BENCH_SHOTS,
       "bank_checkpoint": str(BANK_CHECKPOINT), "bank_reused": BANK_REUSED,
       "output_dir": str(OUTPUT_DIR)}, flush=True)


## 2. Kaggle checkout and live progress

Training uses one visible T4 at a time for predictable memory use. If collection is needed, its two workers each get one GPU. The stream helper forwards tqdm refreshes to the cell immediately.


In [ ]:
import codecs
import subprocess
import sys
import threading

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["ARS_ATTENTION_IMPLEMENTATION"] = ATTENTION_IMPLEMENTATION
REPO_URL = "https://github.com/Abdullah182155/adaptive-residual-steering.git"
REPO_DIR = Path("/kaggle/working/adaptive-residual-steering")
BRANCH = "development"
if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))

def _stream_child(prefix, command, env=None):
    """Stream progress refreshes and return the child status."""
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, env=env, bufsize=0)
    pending = bytearray()
    while True:
        value = process.stdout.read(1)
        if not value: break
        if value in (b"\r", b"\n"):
            if pending:
                print(prefix + codecs.decode(pending, "utf-8", errors="replace"), flush=True)
                pending.clear()
        else:
            pending.extend(value)
    if pending: print(prefix + codecs.decode(pending, "utf-8", errors="replace"), flush=True)
    return process.wait()

def run_live(command, *, env=None, prefix=""):
    code = _stream_child(prefix, command, env)
    if code: raise RuntimeError(f"{prefix}child exited with code {code}: {command}")

def run_parallel_live(jobs):
    results = {}
    def worker(job_id, prefix, command, env): results[job_id] = _stream_child(prefix, command, env)
    threads = [threading.Thread(target=worker, args=(i, *job), daemon=True) for i, job in enumerate(jobs)]
    for thread in threads: thread.start()
    for thread in threads: thread.join()
    failures = [f"job {i} exited with code {value}" for i, value in results.items() if value]
    if failures: raise RuntimeError("; ".join(failures))

print(f"Repository ready: {REPO_DIR} ({BRANCH})", flush=True)


## 3. Runtime, inputs, and phase tracking

Failures are recorded in the phase status dictionary. Missing checkpoints or data skip dependent phases while allowing the final summary to run.


In [ ]:
import json
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PHASE_STATUS = {name: "pending" for name in (
    "runtime", "tests", "inputs", "rollouts", "v2_bootstrap", "norm_calibration",
    "steering_bank", "router", "benchmark")}
try:
    result = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                            capture_output=True, text=True, check=True)
    visible_gpus = [line.strip() for line in result.stdout.splitlines() if line.strip()]
    PHASE_STATUS["runtime"] = "passed" if visible_gpus else "no_gpu_detected"
    print("Visible GPUs:", visible_gpus, flush=True)
except Exception as exc:
    visible_gpus = []
    PHASE_STATUS["runtime"] = f"failed: {type(exc).__name__}: {exc}"
    print(PHASE_STATUS["runtime"], flush=True)
try:
    result = subprocess.run([sys.executable, "-u", "-m", "pytest", "tests/", "-x", "-q"], check=False)
    PHASE_STATUS["tests"] = "passed" if result.returncode == 0 else f"failed_exit_{result.returncode}"
except Exception as exc:
    PHASE_STATUS["tests"] = f"failed: {type(exc).__name__}: {exc}"
print("Tests:", PHASE_STATUS["tests"], flush=True)

if not V2_CHECKPOINT.is_file():
    found = list(Path("/kaggle/input").glob("**/steernet_v2.pt"))
    if found: V2_CHECKPOINT = found[0]
def has_v2_p99_norms(path):
    try:
        calibration = json.loads(path.read_text(encoding="utf-8"))
        layers = calibration.get("layers", {})
        return all(float(layers.get(str(layer), {}).get("p99_norm", 0.0)) > 0 for layer in V2_ROUTE)
    except (OSError, ValueError, TypeError, AttributeError):
        return False

if not has_v2_p99_norms(V2_NORM_CALIBRATION):
    norm_candidates = []
    for filename in ("norms_v2.json", "norms_calibration.json"):
        norm_candidates.extend(Path("/kaggle/input").glob(f"**/{filename}"))
    V2_NORM_CALIBRATION = next((path for path in norm_candidates if has_v2_p99_norms(path)), V2_NORM_CALIBRATION)
PHASE_STATUS["inputs"] = "ready" if V2_CHECKPOINT.is_file() and has_v2_p99_norms(V2_NORM_CALIBRATION) else ("bootstrap_required" if AUTO_BOOTSTRAP_V2_IF_MISSING else "missing_v2_checkpoint_or_norms")
print({"checkpoint": str(V2_CHECKPOINT), "checkpoint_exists": V2_CHECKPOINT.is_file(),
       "v2_norms": str(V2_NORM_CALIBRATION), "v2_norms_exist": V2_NORM_CALIBRATION.is_file()}, flush=True)
def find_complete_rollout_pair(search_roots):
    candidates = {}
    for root in search_roots:
        if not root.exists():
            continue
        for path in root.glob("**/shard[01].jsonl"):
            candidates.setdefault(path.parent.resolve(), {})[path.name] = path
    complete = [pair for pair in candidates.values() if {"shard0.jsonl", "shard1.jsonl"}.issubset(pair)]
    if not complete:
        return []
    complete.sort(key=lambda pair: max(pair[name].stat().st_mtime for name in ("shard0.jsonl", "shard1.jsonl")), reverse=True)
    pair = complete[0]
    return [pair["shard0.jsonl"], pair["shard1.jsonl"]]
ROLLOUT_FILES = find_complete_rollout_pair([ROLLOUT_DIR, Path("/kaggle/working")])
print("Rollout files:", [str(path) for path in ROLLOUT_FILES], flush=True)

PHASE_STATUS["v2_bootstrap"] = "not_needed" if PHASE_STATUS["inputs"] == "ready" else "pending"


## 4. Reuse rollouts or collect if none are attached

Existing verified shards are reused. Only when none are found does the optional two-GPU collection run create new shards in this run directory.


In [ ]:
if len(ROLLOUT_FILES) >= 2:
    PHASE_STATUS["rollouts"] = "reused_existing_shards"
elif AUTO_COLLECT_IF_MISSING:
    try:
        rollout_output = OUTPUT_DIR / "rollouts"
        rollout_output.mkdir(parents=True, exist_ok=True)
        gate0_command = [sys.executable, "-u", "scripts/run_rft_pipeline.py", "--stage", "gate0",
                         "--n_questions", str(GATE0_QUESTIONS), "--k", str(COLLECTION_K),
                         "--temperature", "0.7", "--gate0_nonblocking",
                         "--output_dir", str(rollout_output)]
        gate0_env = os.environ.copy()
        gate0_env["CUDA_VISIBLE_DEVICES"] = "0"
        print(f"Running diagnostic Gate 0 on {GATE0_QUESTIONS} questions before fresh collection", flush=True)
        run_live(gate0_command, env=gate0_env, prefix="[Gate 0] ")
        gate0_stats = json.loads((rollout_output / "gate0_stats.json").read_text(encoding="utf-8"))
        print({key: gate0_stats.get(key) for key in ("n", "pass_at_k_lenient", "greedy_lenient_accuracy", "quality_gate_passed")}, flush=True)
        jobs = []
        for shard, gpu in ((0, "0"), (1, "1")):
            env = os.environ.copy()
            env["CUDA_VISIBLE_DEVICES"] = gpu
            command = [sys.executable, "-u", "scripts/run_rft_pipeline.py", "--stage", "collect",
                       "--n_questions", str(COLLECTION_QUESTIONS), "--k", str(COLLECTION_K),
                       "--temperature", "0.7", "--n_shards", "2", "--shard", str(shard),
                       "--allow_low_yield", "--output_dir", str(rollout_output)]
            jobs.append((f"[GPU{gpu}/shard{shard}] ", command, env))
        run_parallel_live(jobs)
        ROLLOUT_FILES = find_complete_rollout_pair([rollout_output])
        PHASE_STATUS["rollouts"] = "collected" if len(ROLLOUT_FILES) == 2 else "collection_outputs_missing"
    except Exception as exc:
        PHASE_STATUS["rollouts"] = f"failed: {type(exc).__name__}: {exc}"
        print("Rollout phase failed; dependent phases will be skipped:", exc, flush=True)
else:
    PHASE_STATUS["rollouts"] = "missing_and_auto_collection_disabled"
print("Rollout status:", PHASE_STATUS["rollouts"], flush=True)


## 5. Rebuild missing V2 artifacts
When no fixed V2 checkpoint and p99 norms are attached, calibrate and train this comparison arm from the same positive rollout shards. This replacement is recorded separately and does not overwrite the original artifact.

In [ ]:
if PHASE_STATUS["inputs"] == "bootstrap_required":
    if PHASE_STATUS["rollouts"] in {"reused_existing_shards", "collected"} and len(ROLLOUT_FILES) == 2:
        try:
            v2_calibration_command = [sys.executable, "-u", "scripts/calibrate_norms.py",
                                      "--n_samples", str(CALIB_N), "--seed", "42",
                                      "--rollout_files", *map(str, ROLLOUT_FILES),
                                      "--candidate_layers", *map(str, V2_ROUTE),
                                      "--max_length", "768", "--output", str(BOOTSTRAP_V2_NORM_CALIBRATION),
                                      "--device", "cuda"]
            env = os.environ.copy()
            env["CUDA_VISIBLE_DEVICES"] = "0"
            run_live(v2_calibration_command, env=env, prefix="[V2 norms] ")
            if not BOOTSTRAP_V2_NORM_CALIBRATION.is_file():
                raise FileNotFoundError(f"V2 norm calibration was not produced: {BOOTSTRAP_V2_NORM_CALIBRATION}")
            V2_NORM_CALIBRATION = BOOTSTRAP_V2_NORM_CALIBRATION
            if V2_CHECKPOINT.is_file():
                PHASE_STATUS["v2_bootstrap"] = "checkpoint_reused_norms_recalibrated"
            else:
                v2_train_output = OUTPUT_DIR / "v2_bootstrap"
                v2_train_output.mkdir(parents=True, exist_ok=True)
                v2_train_command = [sys.executable, "-u", "scripts/run_rft_pipeline.py",
                                    "--stage", "train", "--skip_collection", "--steernet_arch", "v2",
                                    "--norm_calibration", str(BOOTSTRAP_V2_NORM_CALIBRATION),
                                    "--rollout_dir", str(ROLLOUT_FILES[0].parent),
                                    "--epochs", str(V2_BOOTSTRAP_EPOCHS),
                                    "--checkpoint", str(BOOTSTRAP_V2_CHECKPOINT),
                                    "--output_dir", str(v2_train_output)]
                run_live(v2_train_command, env=env, prefix="[V2 bootstrap] ")
                if not BOOTSTRAP_V2_CHECKPOINT.is_file():
                    raise FileNotFoundError(f"V2 bootstrap did not produce a checkpoint: {BOOTSTRAP_V2_CHECKPOINT}")
                V2_CHECKPOINT = BOOTSTRAP_V2_CHECKPOINT
                PHASE_STATUS["v2_bootstrap"] = "trained_from_rollouts"
            PHASE_STATUS["inputs"] = "bootstrapped"
        except Exception as exc:
            PHASE_STATUS["v2_bootstrap"] = f"failed: {type(exc).__name__}: {exc}"
            PHASE_STATUS["inputs"] = "bootstrap_failed"
            print("V2 bootstrap failed; dependent training will be skipped, but the notebook will continue:", exc, flush=True)
    else:
        PHASE_STATUS["v2_bootstrap"] = "skipped_no_rollouts"
        PHASE_STATUS["inputs"] = "bootstrap_failed"
else:
    PHASE_STATUS["v2_bootstrap"] = "not_needed" if PHASE_STATUS["inputs"] == "ready" else "disabled"
print("V2 bootstrap status:", PHASE_STATUS["v2_bootstrap"], flush=True)


## 5. Calibrate candidate p99 norms

This uses rollout questions only. The original V2 norm file remains separate so the fixed V2 arm retains its calibrated caps.


In [ ]:
if PHASE_STATUS["rollouts"] in {"reused_existing_shards", "collected"} and PHASE_STATUS["inputs"] in {"ready", "bootstrapped"}:
    try:
        command = [sys.executable, "-u", "scripts/calibrate_norms.py", "--n_samples", str(CALIB_N),
                   "--seed", "42", "--rollout_files", *[str(path) for path in ROLLOUT_FILES],
                   "--candidate_layers", *[str(layer) for layer in CANDIDATE_LAYERS],
                   "--max_length", "768", "--output", str(EXPANDED_NORM_CALIBRATION), "--device", "cuda"]
        env = os.environ.copy(); env["CUDA_VISIBLE_DEVICES"] = "0"
        run_live(command, env=env, prefix="[norms] ")
        PHASE_STATUS["norm_calibration"] = "passed" if EXPANDED_NORM_CALIBRATION.is_file() else "output_missing"
    except Exception as exc:
        PHASE_STATUS["norm_calibration"] = f"failed: {type(exc).__name__}: {exc}"
        print(PHASE_STATUS["norm_calibration"], flush=True)
else:
    PHASE_STATUS["norm_calibration"] = "skipped_missing_prerequisites"


## 6. Train the steering bank by route curriculum

The bank is trained in stages: every candidate layer alone first, then validation-screened cross-depth pairs, then triples grown from promising pairs, then quads grown from promising triples. Near-neighbor pairs are mostly pruned, with a small adjacent-pair control kept to detect unexpected synergy. The no-steering route is scored as k=0 and remains available to the router. Only selected adapters update; Phi-2 remains frozen.


In [ ]:
if BANK_REUSED and BANK_CHECKPOINT.is_file():
    PHASE_STATUS["steering_bank"] = "reused_existing_bank"
    print("Reusing existing steering bank:", BANK_CHECKPOINT, flush=True)
elif PHASE_STATUS["norm_calibration"] == "passed" and PHASE_STATUS["inputs"] in {"ready", "bootstrapped"} and ROLLOUT_FILES:
    try:
        command = [sys.executable, "-u", "scripts/run_session3_router.py", "--stage", "train_bank",
                   "--output_dir", str(OUTPUT_DIR), "--candidate_layers", *map(str, CANDIDATE_LAYERS),
                   "--norm_calibration", str(EXPANDED_NORM_CALIBRATION),
                   "--v2_norm_calibration", str(V2_NORM_CALIBRATION), "--v2_checkpoint", str(V2_CHECKPOINT),
                   "--bank_checkpoint", str(BANK_CHECKPOINT), "--rollout_files", *map(str, ROLLOUT_FILES),
                   "--epochs", str(BANK_EPOCHS), "--max_questions", str(MAX_RFT_QUESTIONS),
                   "--route_examples_per_layer", str(BANK_ROUTE_EXAMPLES_PER_LAYER),
                   "--screen_validation_rows", str(BANK_SCREEN_ROWS),
                   "--max_pair_routes", str(MAX_PAIR_ROUTES),
                   "--max_triple_routes", str(MAX_TRIPLE_ROUTES),
                   "--max_quad_routes", str(MAX_QUAD_ROUTES)]
        env = os.environ.copy(); env["CUDA_VISIBLE_DEVICES"] = "0"
        run_live(command, env=env, prefix="[bank] ")
        PHASE_STATUS["steering_bank"] = "passed" if BANK_CHECKPOINT.is_file() else "checkpoint_missing"
    except Exception as exc:
        PHASE_STATUS["steering_bank"] = f"failed: {type(exc).__name__}: {exc}"
        print(PHASE_STATUS["steering_bank"], flush=True)
else:
    PHASE_STATUS["steering_bank"] = "skipped_prerequisite_failure"


## 7. Train the prompt-conditioned router

The router sees prompt tokens only. A training-only teacher scores every route/alpha action in the curated catalog (including k=0) on verified positive completions. A route-value model learns the relative scores of the full action set, rather than learning only one chosen route label. Completion tokens determine teacher scores but are never router inputs. The teacher tests alpha values 1.0, 1.5, and 2.0; route cost is 0.01 NLL per active layer. The production teacher set is capped at 128 prompts to bound the scoring workload; the best router epoch is selected by held-out route NLL.


In [ ]:
if PHASE_STATUS["steering_bank"] in {"passed", "reused_existing_bank"} and ROLLOUT_FILES:
    try:
        command = [sys.executable, "-u", "scripts/run_session3_router.py", "--stage", "train_router",
                   "--output_dir", str(OUTPUT_DIR), "--candidate_layers", *map(str, CANDIDATE_LAYERS),
                   "--norm_calibration", str(EXPANDED_NORM_CALIBRATION),
                   "--v2_norm_calibration", str(V2_NORM_CALIBRATION), "--v2_checkpoint", str(V2_CHECKPOINT),
                   "--bank_checkpoint", str(BANK_CHECKPOINT), "--router_checkpoint", str(ROUTER_CHECKPOINT),
                   "--rollout_files", *map(str, ROLLOUT_FILES), "--epochs", str(ROUTER_EPOCHS),
                   "--max_questions", str(MAX_RFT_QUESTIONS), "--max_router_rows", str(MAX_ROUTER_ROWS),
                   "--route_cost", str(ROUTE_COST)]
        env = os.environ.copy(); env["CUDA_VISIBLE_DEVICES"] = "0"
        run_live(command, env=env, prefix="[router] ")
        PHASE_STATUS["router"] = "passed" if ROUTER_CHECKPOINT.is_file() else "checkpoint_missing"
    except Exception as exc:
        PHASE_STATUS["router"] = f"failed: {type(exc).__name__}: {exc}"
        print(PHASE_STATUS["router"], flush=True)
else:
    PHASE_STATUS["router"] = "skipped_prerequisite_failure"


## 8. Compare the three arms on one paired subset

All arms share the same seeded questions, shot conditions, greedy decoding, and repetition penalty 1.0. The output directory contains arm-level JSONL predictions and paired summaries.


In [ ]:
if PHASE_STATUS["inputs"] in {"ready", "bootstrapped"}:
    try:
        command = [sys.executable, "-u", "scripts/run_session3_router.py", "--stage", "benchmark",
                   "--output_dir", str(OUTPUT_DIR), "--candidate_layers", *map(str, CANDIDATE_LAYERS),
                   "--norm_calibration", str(EXPANDED_NORM_CALIBRATION),
                   "--v2_norm_calibration", str(V2_NORM_CALIBRATION), "--v2_checkpoint", str(V2_CHECKPOINT),
                   "--bank_checkpoint", str(BANK_CHECKPOINT), "--router_checkpoint", str(ROUTER_CHECKPOINT),
                   "--bench_n", str(BENCH_N_TEST), "--bench_seed", "42", "--shots", *map(str, BENCH_SHOTS)]
        env = os.environ.copy(); env["CUDA_VISIBLE_DEVICES"] = "0"
        run_live(command, env=env, prefix="[benchmark] ")
        reports = list(OUTPUT_DIR.glob("session3_gsm8k_*.json"))
        if reports:
            report = json.loads(reports[0].read_text(encoding="utf-8"))
            PHASE_STATUS["benchmark"] = "passed_three_arms" if report.get("router_status") == "included" else "passed_baseline_and_v2_router_unavailable"
        else:
            PHASE_STATUS["benchmark"] = "report_missing"
    except Exception as exc:
        PHASE_STATUS["benchmark"] = f"failed: {type(exc).__name__}: {exc}"
        print(PHASE_STATUS["benchmark"], flush=True)
else:
    PHASE_STATUS["benchmark"] = "skipped_missing_v2_artifacts"
print("Benchmark status:", PHASE_STATUS["benchmark"], flush=True)


## 9. Final status and artifacts

A completed notebook does not imply the router improved accuracy. Keep it only if the paired results show useful improvement over both baseline and fixed V2.


In [ ]:
summary = {
    "phase_status": PHASE_STATUS,
    "run_dir": str(OUTPUT_DIR),
    "v2_checkpoint": str(V2_CHECKPOINT),
    "v2_norm_calibration": str(V2_NORM_CALIBRATION),
    "v2_artifact_provenance": PHASE_STATUS.get("v2_bootstrap"),
    "fixed_route": V2_ROUTE,
    "router_candidates": CANDIDATE_LAYERS,
    "bank_checkpoint": str(BANK_CHECKPOINT) if BANK_CHECKPOINT.is_file() else None,
    "router_checkpoint": str(ROUTER_CHECKPOINT) if ROUTER_CHECKPOINT.is_file() else None,
    "benchmark_reports": [str(path) for path in OUTPUT_DIR.glob("session3_gsm8k_*.json")],
    "per_question_files": [str(path) for path in (OUTPUT_DIR / "generations").glob("*.jsonl")] if (OUTPUT_DIR / "generations").exists() else [],
}
(OUTPUT_DIR / "session3_status.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(json.dumps(summary, indent=2), flush=True)


## 10. One-cell results, statistics, and diagnosis

Run this after the session finishes. It prints one compact report for the baseline, fixed V2, router, paired comparisons, rollout yield, bank route curriculum, router teacher labels, and route diversity. Copy the JSON output and send it as one message; you do not need to download or attach the individual reports.

In [ ]:
import json
import math
import random
from pathlib import Path

import torch


def _read_json(path):
    path = Path(path)
    if not path.is_file():
        return None
    return json.loads(path.read_text(encoding="utf-8"))


def _read_jsonl(path):
    path = Path(path)
    if not path.is_file():
        return []
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]


def _mcnemar_p(wins, losses):
    discordant = wins + losses
    if discordant == 0:
        return 1.0
    tail = sum(math.comb(discordant, index) for index in range(min(wins, losses) + 1))
    return min(1.0, 2.0 * tail / (2 ** discordant))


def _paired_diagnostics(control, treatment, seed=42, bootstrap_samples=2000):
    common = sorted(set(control) & set(treatment))
    if not common:
        return {"n": 0, "status": "no matched question IDs"}
    differences = [int(bool(treatment[key])) - int(bool(control[key])) for key in common]
    wins = sum(value == 1 for value in differences)
    losses = sum(value == -1 for value in differences)
    ties = len(differences) - wins - losses
    rng = random.Random(seed)
    draws = sorted(
        100.0 * sum(rng.choice(differences) for _ in differences) / len(differences)
        for _ in range(bootstrap_samples)
    )
    return {
        "n": len(common),
        "wins": wins,
        "losses": losses,
        "ties": ties,
        "delta_pp": round(100.0 * sum(differences) / len(differences), 3),
        "bootstrap_ci95_pp": [round(draws[int(0.025 * bootstrap_samples)], 3),
                               round(draws[min(bootstrap_samples - 1, int(0.975 * bootstrap_samples))], 3)],
        "mcnemar_p_value": round(_mcnemar_p(wins, losses), 6),
    }


def _arm_index(filename):
    return {int(row.get("source_index", row.get("idx", i))): bool(row.get("lenient", False))
            for i, row in enumerate(_read_jsonl(OUTPUT_DIR / "generations" / filename))}


def _arm_summary(value):
    if not isinstance(value, dict):
        return None
    return {
        "n": value.get("n"),
        "lenient_accuracy_pct": value.get("lenient_accuracy"),
        "strict_accuracy_pct": value.get("strict_accuracy"),
        "marker_accuracy_pct": value.get("marker_accuracy"),
        "format_rate": value.get("format_rate"),
        "guard_trip_rate": value.get("guard_trip_rate"),
        "nonfinite_count": value.get("nonfinite_count"),
    }


def _route_usage_summary(usage):
    if not isinstance(usage, dict):
        return None
    route_counts = usage.get("route_counts", {})
    total = sum(route_counts.values())
    probabilities = [count / total for count in route_counts.values() if total]
    return {
        "questions": total,
        "unique_routes": len(route_counts),
        "top_routes": dict(sorted(route_counts.items(), key=lambda item: item[1], reverse=True)[:10]),
        "k_counts": usage.get("k_counts", {}),
        "layer_counts": usage.get("layer_counts", {}),
        "alpha_counts": usage.get("alpha_counts", {}),
        "route_entropy": round(-sum(p * math.log(p) for p in probabilities), 4) if probabilities else None,
        "most_used_route_share": round(max(probabilities), 4) if probabilities else None,
    }


report = {
    "run_dir": str(OUTPUT_DIR),
    "phase_status": PHASE_STATUS,
    "rollouts": [],
    "training": {},
    "benchmark": {},
    "paired_arm_comparisons": {},
    "diagnosis": [],
}

for rollout_path in ROLLOUT_FILES:
    stats_path = Path(rollout_path).with_name(Path(rollout_path).stem + "_stats.json")
    stats = _read_json(stats_path)
    report["rollouts"].append({
        "file": Path(rollout_path).name,
        "n_questions": stats.get("n") if stats else None,
        "k": stats.get("k") if stats else None,
        "pass_at_k_lenient": stats.get("pass_at_k_lenient") if stats else None,
        "verified_positive_rate": stats.get("positive_sample_rate") if stats else None,
        "stats_available": stats is not None,
    })

bank_state = None
if BANK_CHECKPOINT.is_file():
    bank_state = torch.load(BANK_CHECKPOINT, map_location="cpu", weights_only=True)
    curriculum = bank_state.get("curriculum", {})
    stages = curriculum.get("stages", {})
    report["training"]["bank"] = {
        "candidate_layers": bank_state.get("candidate_layers"),
        "depth_bands": curriculum.get("depth_bands"),
        "k0_validation_nll": curriculum.get("k0_validation_nll"),
        "stages": {},
    }
    for stage_name, stage in stages.items():
        scores = stage.get("validation_token_mean_nll", {})
        selected = stage.get("selected", [])
        ranked = sorted(
            ((route, score) for route, score in scores.items() if route and route != "k0"),
            key=lambda item: item[1],
        )
        report["training"]["bank"]["stages"][stage_name] = {
            "candidate_count": len(stage.get("candidates", [])),
            "selected_count": len(selected),
            "selected_routes": selected,
            "best_scored_routes": [
                {"route": route, "validation_token_mean_nll": round(score, 6)}
                for route, score in ranked[:5]
            ],
        }
    report["training"]["bank"]["router_catalog_count"] = len(curriculum.get("catalog", []))
else:
    report["training"]["bank"] = {"status": "checkpoint missing"}

router_training = _read_json(OUTPUT_DIR / "session3_router_training.json")
if router_training:
    report["training"]["router_teacher"] = {
        "teacher_rows": router_training.get("teacher_labels", {}).get("rows"),
        "teacher_k_counts": router_training.get("teacher_labels", {}).get("k_counts"),
        "teacher_route_counts": router_training.get("teacher_labels", {}).get("route_counts"),
        "teacher_layer_counts": router_training.get("teacher_labels", {}).get("layer_counts"),
        "teacher_alpha_counts": router_training.get("teacher_labels", {}).get("alpha_counts"),
        "route_cost": router_training.get("route_cost"),
        "alpha_values": router_training.get("alpha_values"),
        "best_validation_nll": router_training.get("best_validation_nll"),
        "action_count": router_training.get("action_count"),
        "teacher_target_file": router_training.get("teacher_target_file"),
        "epoch_history": router_training.get("history"),
    }
else:
    report["training"]["router_teacher"] = {"status": "router training report missing"}

benchmark_files = sorted(OUTPUT_DIR.glob("session3_gsm8k_*.json"))
if benchmark_files:
    benchmark_path = benchmark_files[-1]
    benchmark = _read_json(benchmark_path)
    report["benchmark"]["file"] = benchmark_path.name
    report["benchmark"]["test_seed"] = benchmark.get("test_seed")
    report["benchmark"]["source_questions"] = len(benchmark.get("test_source_indices", []))
    for shot, values in benchmark.get("shots", {}).items():
        report["benchmark"][shot + "_shot"] = {
            arm: _arm_summary(values.get(arm)) for arm in ("baseline", "steer_only", "router")
            if values.get(arm) is not None
        }
        report["benchmark"][shot + "_shot"]["router_route_usage"] = _route_usage_summary(
            values.get("router_route_usage")
        )
        for paired_key, label in (("paired_steer_only", "steer_only_vs_baseline"),
                                  ("paired_router", "router_vs_baseline")):
            if values.get(paired_key):
                paired = values[paired_key]
                report["benchmark"][shot + "_shot"][label] = {
                    key: paired.get(key) for key in (
                        "fixed_wins", "fixed_losses", "ties", "delta_pp", "ci95_pp",
                        "mcnemar_p_value", "identical_text_rate", "null_intervention_warning",
                    ) if key in paired
                }

        baseline = _arm_index(f"gsm8k_{shot}shot_baseline.jsonl")
        fixed = _arm_index(f"gsm8k_{shot}shot_steer_only.jsonl")
        routed = _arm_index(f"gsm8k_{shot}shot_router.jsonl")
        report["paired_arm_comparisons"][shot + "_shot"] = {
            "fixed_v2_vs_baseline": _paired_diagnostics(baseline, fixed),
            "router_vs_baseline": _paired_diagnostics(baseline, routed),
            "router_vs_fixed_v2": _paired_diagnostics(fixed, routed),
        }
else:
    report["benchmark"] = {"status": "benchmark report missing"}

for phase, status in PHASE_STATUS.items():
    if any(token in str(status).lower() for token in ("failed", "skipped", "missing", "pending")):
        report["diagnosis"].append(f"{phase}: {status}")

for shot, comparisons in report["paired_arm_comparisons"].items():
    router_delta = comparisons["router_vs_fixed_v2"].get("delta_pp")
    if router_delta is not None:
        report["diagnosis"].append(
            f"{shot}: router vs fixed V2 paired delta is {router_delta:+.2f} pp; "
            "use this paired comparison, with its interval and p-value, to judge the router."
        )

usage_entries = [
    values.get("router_route_usage") for key, values in report["benchmark"].items()
    if key.endswith("_shot") and isinstance(values, dict)
]
if usage_entries and not any(item and item.get("questions", 0) for item in usage_entries):
    report["diagnosis"].append("Router benchmark route-usage metadata is empty; check router prefill and arm selection.")
elif usage_entries:
    report["diagnosis"].append(
        "Router diversity is reported as observed routes, k counts, layer counts, route entropy, "
        "and the most-used-route share; interpret diversity together with paired accuracy."
    )
if not report["diagnosis"]:
    report["diagnosis"].append("All recorded phases completed; compare paired deltas and confidence intervals before selecting a model.")

DIAGNOSTICS_PATH = OUTPUT_DIR / "session3_diagnostics_for_review.json"
DIAGNOSTICS_PATH.write_text(json.dumps(report, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print("Copy this complete report and send it for review:", flush=True)
print(json.dumps(report, indent=2, ensure_ascii=False), flush=True)
print(f"Saved copy: {DIAGNOSTICS_PATH}", flush=True)
